# [센서 실습 28] GPS 위치 센서 – NEO-6M GPS 모듈

GPS 모듈 **NEO-6M**이 UART로 보내는 **NMEA 문장($GPGGA)** 을 직접 해석하여 위도·경도를 구하고, 이동 거리·속도 계산과 **지오펜스(가상 울타리)** 도착 알림을 만듭니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
GPS 위성은 정확한 시각이 담긴 전파를 계속 보냅니다. 수신기는 **4개 이상의 위성** 신호가 도착한 시간 차이로 자신의 위치를 계산합니다(삼변측량).
- GPS 모듈은 계산 결과를 **NMEA 0183** 형식의 글자(문장)로 1초마다 보냄
- 예: `$GPGGA,123519,3733.990,N,12658.680,E,1,08,0.9,38.0,M,0.0,M,,*47`

| 항목 | 의미 |
|---|---|
| `3733.990,N` | 위도 37도 33.990분 → 37 + 33.990/60 = **37.56650°** |
| `12658.680,E` | 경도 126도 58.680분 → **126.97800°** |
| `1` | 위치 고정 품질 (0 = 위치 없음) |
| `08` | 사용한 위성 수 |
| `0.9` | HDOP (작을수록 정확) |
| `*47` | `$`와 `*` 사이 글자들을 XOR 한 **체크섬** |

- 전원을 켠 뒤 위치를 처음 잡을 때까지(**TTFF**) 수십 초 ~ 몇 분 걸림, 실내에서는 거의 안 됨
- 두 지점 사이 거리는 지구 곡률을 고려한 **하버사인(haversine) 공식**으로 계산

### 부품 사양
| 항목 | 내용 |
|---|---|
| 채널 | 50채널, GPS L1 |
| 위치 정확도 | 약 2.5m (개활지) |
| 처음 위치 잡기 | 콜드 스타트 약 27초 이상 |
| 출력 | NMEA 0183, 1Hz |
| 통신 | UART 9600bps |
| 전원 | 3.3 ~ 5V |

## 2. 라즈베리파이 연결 방법
**준비물**: NEO-6M 모듈 + 세라믹 안테나, 점퍼선 4개 (창가나 실외에서 실습)

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| NEO-6M VCC / GND | 3.3V / GND |  |
| NEO-6M TX | GPIO15 / RXD (10번 핀) | 모듈 → Pi |
| NEO-6M RX | GPIO14 / TXD (8번 핀) | Pi → 모듈 (설정 변경 시) |



**인터페이스 활성화**
- **UART**: `sudo raspi-config` → Interface Options → **Serial Port** → 로그인 셸: 아니요, 시리얼 하드웨어: 예

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install pyserial
# 확인: cat /dev/serial0   ($GPGGA … 문장이 흘러나오면 성공)
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
NMEA 체크섬 계산과 `$GPGGA` 문장 해석, 두 좌표 사이 거리(하버사인) 함수입니다. 외부 라이브러리 없이 직접 구현합니다.

In [ ]:
def nmea_checksum(body):
    c = 0
    for ch in body:
        c ^= ord(ch)
    return c

def dm_to_deg(dm, hemi, deg_len):
    deg = int(dm[:deg_len]) + float(dm[deg_len:]) / 60
    return -deg if hemi in ("S", "W") else deg

def parse_gga(sentence):
    """$GPGGA 문장 → 위치 딕셔너리 (체크섬 오류·위치 없음이면 None)"""
    s = sentence.strip()
    if not s.startswith("$") or "*" not in s:
        return None
    body, cs = s[1:].split("*", 1)
    if int(cs[:2], 16) != nmea_checksum(body):
        return None
    f = body.split(",")
    if not f[0].endswith("GGA") or f[6] in ("", "0"):
        return None
    return {"lat": round(dm_to_deg(f[2], f[3], 2), 6), "lon": round(dm_to_deg(f[4], f[5], 3), 6),
            "sats": int(f[7]), "hdop": float(f[8]), "alt_m": float(f[9])}

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

print(parse_gga("$GPGGA,123519,3733.990,N,12658.680,E,1,08,0.9,38.0,M,0.0,M,,*47"))

### 4-2. 실제 센서 클래스
시리얼 포트에서 한 줄씩 읽어 `$GPGGA`(또는 `$GNGGA`) 문장이 나오면 해석합니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealNEO6M:
    def __init__(self, port="/dev/serial0"):
        import serial
        self.ser = serial.Serial(port, baudrate=9600, timeout=2)

    def read(self, max_lines=20):
        for _ in range(max_lines):
            line = self.ser.readline().decode("ascii", errors="ignore")
            if "GGA" in line:
                pos = parse_gga(line)
                return pos if pos else {"lat": None, "lon": None, "sats": 0, "hdop": None, "alt_m": None}
        return {"lat": None, "lon": None, "sats": 0, "hdop": None, "alt_m": None}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 항목 | 가정한 값 | 근거 |
|---|---|---|
| 경로 | 서울시청 → 덕수궁 대한문 → 세종대로 → 광화문 | 도보 견학 |
| 걷는 속도 | 1.3 m/s | 성인 보통 걸음 |
| 처음 20초 | 위치 없음(품질 0) | 콜드 스타트 TTFF |
| 위치 오차 | 약 ±3 m, 천천히 떠다니는 오차 (HDOP 0.9~1.6) | NEO-6M 개활지 정확도, 실제 GPS 오차는 갑자기 튀지 않고 서서히 변함 |
| 체크섬 오류 | 2% | 시리얼 수신 오류 |

가상 센서는 경로 위의 위치로 **체크섬까지 맞춘 진짜 `$GPGGA` 문장**을 만들어 해석기에 넘깁니다.

In [ ]:
class VirtualNEO6M:
    ROUTE = [(37.566535, 126.977969), (37.565700, 126.976500), (37.568300, 126.976800),
             (37.571300, 126.976900), (37.575800, 126.976800)]      # 시청 → 대한문 → 세종대로 → 광화문
    SPEED = 1.3

    def __init__(self, dt=2.0, seed=28):
        self.dt, self.t = dt, 0.0
        self.rng = np.random.default_rng(seed)
        seg = [haversine_m(*a, *b) for a, b in zip(self.ROUTE, self.ROUTE[1:])]
        self.cum = np.concatenate([[0], np.cumsum(seg)])
        self.err = np.zeros(2)                                     # 천천히 변하는 위치 오차(m)

    def _position(self, dist):
        dist = min(dist, self.cum[-1])
        i = min(np.searchsorted(self.cum, dist, side="right") - 1, len(self.ROUTE) - 2)
        r = (dist - self.cum[i]) / (self.cum[i + 1] - self.cum[i])
        (a1, o1), (a2, o2) = self.ROUTE[i], self.ROUTE[i + 1]
        return a1 + (a2 - a1) * r, o1 + (o2 - o1) * r

    @staticmethod
    def make_gga(utc, lat, lon, fix, sats, hdop, alt):
        lat_s = f"{int(abs(lat)):02d}{(abs(lat) % 1) * 60:07.4f},{'N' if lat >= 0 else 'S'}"
        lon_s = f"{int(abs(lon)):03d}{(abs(lon) % 1) * 60:07.4f},{'E' if lon >= 0 else 'W'}"
        body = f"GPGGA,{utc},{lat_s},{lon_s},{fix},{sats:02d},{hdop:.1f},{alt:.1f},M,18.5,M,,"
        return f"${body}*{nmea_checksum(body):02X}"

    def read(self):
        walk = max(0.0, self.t - 20)                                   # 20초 뒤부터 출발
        lat, lon = self._position(walk * self.SPEED)
        self.err = 0.9 * self.err + self.rng.normal(0, 1.3, 2)        # 오차는 천천히 변함 (표준편차 약 3m)
        lat += self.err[0] / 111000
        lon += self.err[1] / 88000
        fix = 0 if self.t < 20 else 1                                  # 콜드 스타트
        utc = f"{(3 + int(self.t // 3600)):02d}{int(self.t % 3600 // 60):02d}{int(self.t % 60):02d}"
        s = self.make_gga(utc, lat, lon, fix, 0 if fix == 0 else int(self.rng.integers(6, 11)),
                          self.rng.uniform(0.9, 1.6), 38 + self.rng.normal(0, 2))
        if self.rng.random() < 0.02:
            s = s[:-1] + ("0" if s[-1] != "0" else "1")                # 체크섬 오류
        self.t += self.dt
        pos = parse_gga(s)
        return dict(pos, nmea=s) if pos else {"lat": None, "lon": None, "sats": 0, "hdop": None, "alt_m": None, "nmea": s}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualNEO6M(dt=2.0)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
2초 간격으로 20분 동안 기록합니다. 가상 센서는 비교를 위해 원래 NMEA 문장(`nmea`)도 함께 기록합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 600               # 측정 횟수
INTERVAL = 2.0               # 측정 간격(초)

sensor = RealNEO6M() if USE_REAL_SENSOR else VirtualNEO6M(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
1. 위치가 없는 값(콜드 스타트·체크섬 오류)을 제외합니다.
2. 하버사인 공식으로 **이동 거리와 평균 속도**를 구합니다.
3. 광화문 중심 **반경 80m 지오펜스**에 들어온 시각을 찾아 도착 알림을 보냅니다.

In [ ]:
valid = df.dropna(subset=["lat"]).reset_index(drop=True)
first_fix = valid.time_s.iloc[0]
print(f"처음 위치를 잡은 시각(TTFF): {first_fix:.0f}초, 제외한 값 {len(df) - len(valid)}개")
def path_length(lat, lon, min_step=0.0):
    d = haversine_m(lat[:-1], lon[:-1], lat[1:], lon[1:])
    return d[d >= min_step].sum()

raw_total = path_length(valid.lat.values, valid.lon.values)
valid["lat_s"] = valid.lat.rolling(5, center=True, min_periods=1).mean()   # 이동 평균으로 오차 줄이기
valid["lon_s"] = valid.lon.rolling(5, center=True, min_periods=1).mean()
total = path_length(valid.lat_s.values, valid.lon_s.values, min_step=0.5)  # 0.5m 미만 흔들림은 정지로 봄
moving = valid[haversine_m(valid.lat_s, valid.lon_s, valid.lat_s.shift(), valid.lon_s.shift()).fillna(0) >= 0.5]
moving_s = len(moving) * INTERVAL
print(f"원시 좌표로 계산한 거리 : {raw_total:.0f} m  (위치 오차가 쌓여 실제보다 길게 나옴)")
print(f"평활화 후 이동 거리     : {total:.0f} m, 걸은 시간 {moving_s / 60:.1f}분, 평균 속도 {total / moving_s:.2f} m/s")

GATE = (37.575800, 126.976800)          # 광화문
valid["to_gate_m"] = haversine_m(valid.lat, valid.lon, *GATE)
inside = valid[valid.to_gate_m < 80]
if len(inside):
    r = inside.iloc[0]
    print(f"📍 {r.time_s / 60:.1f}분: 광화문 지오펜스(반경 80m) 도착 → '도착했습니다' 알림 전송")
print("\n가상 센서가 만든 NMEA 예:", df["nmea"].iloc[30] if "nmea" in df else "")

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(valid.lon, valid.lat, ".", color="gray", ms=3, label="GPS 원시 좌표")
ax.plot(valid.lon_s, valid.lat_s, "k-", lw=1.5, label="평활화 경로")
ax.plot(valid.lon.iloc[0], valid.lat.iloc[0], "ks", ms=9, label="출발 (서울시청)")
th = np.linspace(0, 2 * np.pi, 100)
ax.plot(GATE[1] + 80 / 88000 * np.cos(th), GATE[0] + 80 / 111000 * np.sin(th), "k--", label="지오펜스 80m (광화문)")
ax.set_aspect(1 / np.cos(np.radians(37.57))); ax.ticklabel_format(useOffset=False)
ax.set_xlim(126.970, 126.984)
ax.set_xlabel("경도"); ax.set_ylabel("위도"); ax.set_title("NEO-6M GPS 이동 경로 (서울시청 → 광화문)")
ax.legend(loc="center right", fontsize=9); ax.grid(alpha=0.3)
plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "GPS위치센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "NEO-6M GPS 모듈",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/gps", payload)
# client.disconnect()

## 직접 해 보기
- 위치 오차를 ±10m로 키우면 이동 거리 계산이 얼마나 달라지나요? 오차를 줄이는 방법은?
- 학교 정문 좌표로 지오펜스를 바꿔 등교 알림을 만들어 보세요.
- `$GPRMC` 문장에는 속도와 날짜가 들어 있습니다. 형식을 조사해 해석 함수를 만들어 보세요.